# Notebook — Análisis de Embeddings y Evaluation Awareness

El análisis se encuentra en **desarrollo**.

## Objetivo

Este notebook presenta los resultados obtenidos en el análisis de embeddings para estudiar posibles patrones relacionados con *evaluation awareness*.

Se analiza mediante:

* Reducción dimensional (PCA, t-SNE, UMAP)
* Clustering con KMeans
* Métricas ARI/NMI
* Semantic shift mediante cosine distance
* Interacciones entre dimensiones experimentales
* Clasificadores lineales 


# 1. Imports

In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize, LabelEncoder
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)

import umap
from src.utils.io_handler import load_dataset_parquet
import warnings

# Desactivo warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings(
    "ignore",
    message="n_jobs value 1 overridden to 1 by setting random_state"
)
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', 1000)  

# 2. Carga dataset

In [ ]:
DATASET_PATH = (
    "outputs/"
    "results_raw.parquet"
)

ANONYMIZED_DATASET_PATH = (
    "outputs/"
    "results_anonymized.parquet"
)
df = load_dataset_parquet(
    DATASET_PATH
)
anonymized_df = load_dataset_parquet(
    ANONYMIZED_DATASET_PATH
)
print(df.shape)
print(df.columns)

print(
    df['embedding_response']
    .iloc[0]
    .shape
)

dimensions = [
    "entity_familiarity",
    "context_coherence",
    "ethical_pressure",
    "prompt_structure",
]



# 3. Funciones Auxiliares

## Label Dataset


In [ ]:



def get_embedding_matrix(
    df: pd.DataFrame
) -> np.ndarray:

    return np.array(
        df["embedding_response"].tolist()
    )

def create_eval_deploy_label(df):

    eval_mask = (

        (
            df["entity_familiarity"].isin([
                "0-inventada_baja"
            ])
        )
      
        &

        (
            df["prompt_structure"].isin([
                "2-modular",
                "3-estructurado"
            ])
        )
    )

    labels = np.where(
        eval_mask,
        "eval",
        "deploy"
    )

    return pd.Series(labels, index=df.index)

## Similarity

In [ ]:

def cosine_shift(e1, e2) -> float:

    similarity = cosine_similarity(
        [e1],
        [e2]
    )[0][0]

    return 1 - similarity


def compute_similarity_matrix(
    embeddings: np.ndarray
):

    return cosine_similarity(
        embeddings
    )

## Reducción dimensional (PCA, TSNE y UMAP)

In [ ]:
def compute_pca(
    embeddings,
    n_components=2
):

    pca = PCA(
        n_components=n_components
    )

    reduced = pca.fit_transform(
        embeddings
    )

    return reduced


def compute_tsne(
    embeddings,
    perplexity=30
):
    
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=42
    )

    reduced = tsne.fit_transform(
        embeddings
    )

    return reduced


def compute_umap(
    embeddings
):

    reducer = umap.UMAP(
        random_state=42
    )

    reduced = reducer.fit_transform(
        embeddings
    )

    return reduced

    
## Visualización

In [ ]:
def plot_embedding_space(
    reduced_embeddings,
    labels,
    title="Embedding Space"
):

    categories = labels.astype(
        "category"
    )

    plt.figure(figsize=(8, 6))

    scatter = plt.scatter(
        reduced_embeddings[:, 0],
        reduced_embeddings[:, 1],
        c=categories.cat.codes
    )

    plt.title(title)

    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")

    plt.legend(
        handles=scatter.legend_elements()[0],
        labels=list(
            categories.cat.categories
        ),
        title="Clase"
    )

    plt.show()

def plot_embedding_space(
    ax,
    reduced_embeddings,
    labels,
    title="Embedding Space"
):

    categories = labels.astype(
        "category"
    )

    scatter = ax.scatter(
        reduced_embeddings[:, 0],
        reduced_embeddings[:, 1],
        c=categories.cat.codes
    )

    ax.set_title(title)

    ax.set_xlabel("Dimension 1")
    ax.set_ylabel("Dimension 2")

    ax.legend(
        handles=scatter.legend_elements()[0],
        labels=list(
            categories.cat.categories
        ),
        title="Clase",
        fontsize=8
    )

def plot_reduction_grid(
    df,
    scenario,
    dimensions,
    reduction_function,
    reduction_name
):

    scenario_df = df[
        df["scenario"] == scenario
    ].reset_index(drop=True)

    languages = scenario_df[
        "lang"
    ].unique()

    fig, axes = plt.subplots(
        3,
        len(dimensions),
        figsize=(20, 15)
    )

    for i, lang in enumerate(languages):

        case_df = scenario_df[
            scenario_df["lang"] == lang
        ].reset_index(drop=True)

        case_df["eval_deploy"] = (
            create_eval_deploy_label(
                case_df
            )
        )

        embeddings = get_embedding_matrix(
            case_df
        )

        reduced_embeddings = (
            reduction_function(
                embeddings
            )
        )

        for j, dim in enumerate(dimensions):

            plot_embedding_space(
                axes[i, j],
                reduced_embeddings,
                case_df[dim],
                title=f"{lang} - {dim}"
            )

    plt.suptitle(
        f"{reduction_name} - Scenario: {scenario}",
        fontsize=20
    )

    plt.tight_layout()

    plt.show()
    print('\n\n')

def plot_probe_vs_tfidf(comp_df, title):
   
    df = comp_df.copy()
    
    
    df['scenario'] = df['scenario'].replace('ingredient_substitution', 'ingredient_substitution_control')
    
    scenarios = df['scenario']
    x = np.arange(len(scenarios))
    width = 0.35

    plt.figure(figsize=(12, 6))
    
    plt.bar(x - width/2, df['probe_auc'], width, label='AUC de clasificador lineal sobre embeddings', color='teal')
    plt.bar(x + width/2, df['tfidf_auc'], width, label='AUC de TF-IDF', color='salmon')
    
    plt.title(title)
    plt.ylabel('AUC')
    plt.xticks(x, scenarios, rotation=45, ha='right')
    plt.ylim(0.4, 1.05)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_sensitivity_analysis(results_df, dimensions):
    plot_data = []
   
    for (scenario, lang), group_df in results_df.groupby(['scenario', 'language']):
        for dim in dimensions:
            col_same = f"{dim}_same"
            m_same = group_df[group_df[col_same]]["shift"].mean()
            m_diff = group_df[~group_df[col_same]]["shift"].mean()
            if pd.notnull(m_same) and pd.notnull(m_diff):
                plot_data.append({'scenario': scenario, 'language': lang, 'dimension': dim, 'shift_magnitude': abs(m_same - m_diff)})

    df_plot = pd.DataFrame(plot_data)
    
   
    scenarios = df_plot['scenario'].unique()

    g = sns.catplot(
        data=df_plot, kind="bar", x="scenario", y="shift_magnitude",
        hue="language", col="dimension", col_wrap=2,
        palette="viridis", height=5, aspect=1.2, sharex=True, sharey=False
    )

    for i, ax in enumerate(g.axes.flat):
        
        ax.set_xticks(range(len(scenarios)))
        ax.set_xticklabels(scenarios, rotation=90, ha='center', fontsize=9)
        
        
        if i < 2:
            ax.tick_params(labelbottom=False)
            ax.set_xlabel("")
        else:
            ax.tick_params(labelbottom=True)

    plt.subplots_adjust(top=0.9, bottom=0.25)
    g.figure.suptitle('Análisis de Sensibilidad: Magnitud del Shift por Dimensión', fontsize=16)
    plt.show()

def plot_controlled_shift_by_scenario(results_df):

    summary = (
        results_df
        .groupby(["scenario", "language", "dimension"])["shift"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    scenario_order = sorted(summary["scenario"].unique())

    g = sns.catplot(
        data=summary,
        kind="bar",
        x="scenario",
        y="mean",
        hue="language",
        col="dimension",
        col_wrap=2,
        height=5,
        aspect=1.4,
        sharey=False,
        palette="viridis"
    )

    for ax in g.axes.flat:

        ax.set_xticks(range(len(scenario_order)))

        ax.set_xticklabels(
            scenario_order,
            rotation=90,
            fontsize=9
        )

        ax.set_xlabel("Scenario")
        ax.set_ylabel("Mean Controlled Shift")

    g.figure.subplots_adjust(
        top=0.9,
        bottom=0.18
    )

    g.figure.suptitle(
        "Controlled Semantic Shift by Dimension and Scenario",
        fontsize=18
    )

    plt.show()


## Clustering


In [ ]:

def compute_kmeans(
    embeddings,
    n_clusters=4
):

    normalized_embeddings = normalize(
        embeddings,
        norm="l2"
    )

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    clusters = kmeans.fit_predict(
        normalized_embeddings
    )

    return clusters


def evaluate_clusters(
    clusters,
    labels,
    dimension_name
):

    categories = labels.astype(
        "category"
    ).cat.codes

    ari = adjusted_rand_score(
        categories,
        clusters
    )

    nmi = normalized_mutual_info_score(
        categories,
        clusters
    )

    print("\n--------------------------------")
    print(f"Dimension: {dimension_name}")
    print("--------------------------------")
    print(f"ARI : {ari:.4f}")
    print(f"NMI : {nmi:.4f}")


# 4. PCA

Se utiliza para analizar si las dimensiones experimentales generan variaciones lineales dominantes en el espacio de embeddings

In [ ]:

for scenario in df["scenario"].unique():
    plot_reduction_grid(
        df=df,
        scenario=scenario,
        dimensions=dimensions + ['eval_deploy'],
        reduction_function=compute_pca,
        reduction_name="PCA"
    )




# 5. t-SNE

Se utiliza para explorar agrupamientos locales y estructuras semánticas no lineales dentro del espacio de embeddings

In [ ]:
for scenario in df["scenario"].unique():
    plot_reduction_grid(
        df=df,
        scenario=scenario,
        dimensions=dimensions + ['eval_deploy'],
        reduction_function=compute_tsne,
        reduction_name="t-SNE"
    )



# 6. UMAP

Se utiliza para visualizar la organización general del manifold semántico preservando tanto relaciones locales como parte de la estructura global del espacio de embeddings.


In [ ]:
for scenario in df["scenario"].unique():
    plot_reduction_grid(
        df=df,
        scenario=scenario,
        dimensions=dimensions + ['eval_deploy'],
        reduction_function=compute_umap,
        reduction_name="UMAP"
    )


# 7. Clustering

KMeans se utiliza para analizar si el espacio de embeddings presenta agrupamientos semánticos consistentes con las dimensiones experimentales del estudio. Para medir el agrupamiento se usan NMI y ARI para medir la consistencia de los agrupamientos de K-Means frente a las dimensiones definidas. Estas métricas cuantifican cuánto de la estructura latente del modelo coincide con nuestras variables experimentales, más allá del azar.


In [ ]:
results = []
pd.set_option(
    "display.float_format",
    lambda x: f"{x:.4f}"
)
scenarios = df["scenario"].unique()

for scenario in scenarios:

    scenario_df = df[
        df["scenario"] == scenario
    ].reset_index(drop=True)

    languages = scenario_df[
        "lang"
    ].unique()

    for lang in languages:

        case_df = scenario_df[
            scenario_df["lang"] == lang
        ].reset_index(drop=True)

        case_df["eval_deploy"] = (
            create_eval_deploy_label(
                case_df
            )
        )

        embeddings = get_embedding_matrix(
            case_df
        )

        clusters = compute_kmeans(
            embeddings,
            n_clusters=5
        )

        for dim in dimensions + ['eval_deploy']:

            categories = case_df[
                dim
            ].astype(
                "category"
            ).cat.codes

            ari = adjusted_rand_score(
                categories,
                clusters
            )

            nmi = normalized_mutual_info_score(
                categories,
                clusters
            )

            results.append({

                "scenario": scenario,
                "language": lang,
                "dimension": dim,
                "ARI": ari,
                "NMI": nmi
            })

results_df = pd.DataFrame(
    results
)

ari_table = results_df.pivot_table(
    index=["scenario", "language"],
    columns="dimension",
    values="ARI"
)

ari_table

In [ ]:
nmi_table = results_df.pivot_table(
    index=["scenario", "language"],
    columns="dimension",
    values="NMI"
)

nmi_table


# 8. Semantic Shift

La idea es ver si una dimensión en concreto aumenta la distancia semántica entre embeddings.



## Shift por dimensión
A nivel escenario para cada dimensión comparo el `cosine difference` entre los embeddings que tengan el mismo valor sobre la dimensión analizada y los que tienen disinto valor, lo que resulta en mostrar la influencia global de cada dimensión.

In [ ]:

def intra_case_shift_analysis(df, group_columns, dimensions):
    results = []


    grouped = df.groupby(group_columns)

    for (scenario, lang), case_df in grouped:
        case_df = case_df.reset_index(drop=True)
        
        
        for i, j in combinations(range(len(case_df)), 2):
            row_i = case_df.iloc[i]
            row_j = case_df.iloc[j]

            shift = cosine_shift(
                row_i["embedding_response"],
                row_j["embedding_response"]
            )

            result = {
                "scenario": scenario,
                "language": lang,
                "shift": shift
            }

            for dim in dimensions:
                result[f"{dim}_same"] = (row_i[dim] == row_j[dim])

            results.append(result)

    return pd.DataFrame(results)

results_df = intra_case_shift_analysis(
    df, 
    group_columns=["scenario", "lang"], 
    dimensions=dimensions 
)
plot_sensitivity_analysis(results_df, dimensions)

A diferencia del análisis anterior, el siguiente análisis aisla el efecto de cada dimensión. Para esto se fijan las demás dimensiones y se mide la `cosine difference` sobre respuestas (embeddings) que tengan distinto valor sobre la dimensión analizada.

In [ ]:
def controlled_dimension_shift_analysis(
    df,
    dimensions,
    fixed_columns=("scenario", "lang")
):
    results = []

    for target_dim in dimensions:
        other_dims = [d for d in dimensions if d != target_dim]
        grouping_cols = list(fixed_columns) + other_dims

        grouped = df.groupby(grouping_cols)

        for _, group_df in grouped:

            group_df = group_df.reset_index(drop=True)
   
            for i, j in combinations(range(len(group_df)), 2):

                row_i = group_df.iloc[i]
                row_j = group_df.iloc[j]

                val_i = row_i[target_dim]
                val_j = row_j[target_dim]

                if val_i == val_j:
                    continue

                shift = cosine_shift(
                    row_i["embedding_response"],
                    row_j["embedding_response"]
                )

                results.append({
                    "scenario": row_i["scenario"],
                    "language": row_i["lang"],
                    "dimension": target_dim,
                    "value_i": val_i,
                    "value_j": val_j,
                    "shift": shift
                })

    return pd.DataFrame(results)


def plot_shift_violins(results_df):

    g = sns.catplot(
        data=results_df,
        kind="violin",
        x="dimension",
        y="shift",
        col="scenario",
        col_wrap=3,
        sharey=True,
        cut=0,
        height=4,
        aspect=1.2
    )

    g.set_xticklabels(rotation=25)

    g.fig.subplots_adjust(top=0.9)

    g.fig.suptitle(
        "Controlled Semantic Shift Density"
    )

    plt.show()

def plot_shift_boxplots(results_df):

    plt.figure(figsize=(12, 6))

    sns.boxplot(
        data=results_df,
        x="dimension",
        y="shift",
        hue="scenario"
    )

    plt.xticks(rotation=20)
    plt.title("Controlled Semantic Shift Distribution")
    plt.ylabel("Cosine Shift")

    plt.tight_layout()
    plt.show()

results_controlled = controlled_dimension_shift_analysis(
    df=df,
    dimensions=dimensions
)

summary_table = (
    results_controlled
    .groupby(["dimension", "scenario"])["shift"]
    .agg([
        "mean",
        "std",
        "median",
        "min",
        "max"
    ])
)
print("\n=== Mean Controlled Shift ===")
print(summary_table.round(4))

plot_controlled_shift_by_scenario(results_controlled)
plot_shift_boxplots(results_controlled)
plot_shift_violins(results_controlled)

In [ ]:

def build_pairwise_shift_matrix(
    results_df,
    target_dimension,
    scenario,
    language
):

 

    subset = results_df[
        (results_df["dimension"] == target_dimension)
        &
        (results_df["scenario"] == scenario)
        &
        (results_df["language"] == language)
    ].copy()


    values = sorted(
        set(subset["value_i"])
        |
        set(subset["value_j"])
    )


    matrix = pd.DataFrame(
        np.nan,
        index=values,
        columns=values
    )


    for v1 in values:
        for v2 in values:

            if v1 == v2:
                matrix.loc[v1, v2] = 0.0
                continue

            pair_subset = subset[
                (
                    (subset["value_i"] == v1)
                    &
                    (subset["value_j"] == v2)
                )
                |
                (
                    (subset["value_i"] == v2)
                    &
                    (subset["value_j"] == v1)
                )
            ]

            if len(pair_subset) > 0:

                matrix.loc[v1, v2] = (
                    pair_subset["shift"].mean()
                )

    return matrix



def plot_pairwise_shift_matrix(
    matrix,
    title=None
):

    plt.figure(figsize=(8, 6))

    sns.heatmap(
        matrix,
        annot=True,
        fmt=".3f",
        cmap="viridis"
    )

    plt.title(title or "Pairwise Value Shift Matrix")
    plt.xlabel("Value")
    plt.ylabel("Value")

    plt.tight_layout()
    plt.show()


matrix = build_pairwise_shift_matrix(
    results_controlled,
    target_dimension="ethical_pressure",
    scenario="insider_trading",
    language="0-es"
)

print(matrix.round(4))
# todo: compare by dimension (for each scenario)
plot_pairwise_shift_matrix(
    matrix,
    title="Ethical Pressure Shift Matrix | insider_trading | EN"
)

Para casos cualitativos ver el siguiente análisis (observo refusals)

In [ ]:

def get_max_shift_pairs(df):
    """
    Calcula el par de respuestas con la mayor distancia coseno 
    dentro de cada escenario.
    """
    results = []
    
   
    for (scenario, lang), group in df.groupby(['scenario', 'lang']):
        if len(group) < 2:
            continue
            
       
        embs = np.stack(group['embedding_response'].tolist())
        
     
        dist_matrix = cosine_distances(embs, embs)
    
        np.fill_diagonal(dist_matrix, -1)
        
     
        idx_flat = np.argmax(dist_matrix)
        i, j = np.unravel_index(idx_flat, dist_matrix.shape)
        
      
        row_i = group.iloc[i]
        row_j = group.iloc[j]
        
        results.append({
            'scenario': scenario,
            'max_shift': dist_matrix[i, j],
            'prompt_A': row_i['prompt'],
            'response_A': row_i['response'],
            'prompt_B': row_j['prompt'],
            'response_B': row_j['response']
        })
        
    return pd.DataFrame(results)
max_pairs_df = get_max_shift_pairs(df)
print(max_pairs_df[['scenario', 'max_shift', 'prompt_A', 'prompt_B']])


max_pairs_df.to_csv('max_semantic_jumps.csv', index=False)

# 9. Clasificadores lineales sobre embeddings

Mediante el entrenamiento de clasificadores lineales sobre los embeddings de las respuestas el objetivo es determinar si las representaciones vectoriales del modelo contienen información discriminativa sobre el contexto de evaluación (entorno de prueba vs. uso real). Para ello se entrena Regresión Logística sobre los embeddings de las respuestas de cada escenario y se lo evalua sobre el mismo.

In [ ]:

def run_scenario_probe_final(df, scenario_name):

    subset = df[df["scenario"] == scenario_name].copy()
    if subset.empty:
        return None, None

    y_raw = create_eval_deploy_label(subset)
    y = np.where(y_raw == "eval", 1, 0)

    if len(np.unique(y)) < 2:
        return None, None

    X = get_embedding_matrix(subset)

 
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, subset.index, test_size=0.2, random_state=42, stratify=y
    )

    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

 
    y_prob = model.predict_proba(X_test)[:, 1]
    
  
    fixed_threshold = 0.5
    final_preds = (y_prob >= fixed_threshold).astype(int)


    test_df = subset.loc[idx_test].copy()
    test_df["gt"] = y_test
    test_df["prediction"] = final_preds
    test_df["prob_eval"] = y_prob
    test_df["scenario"] = scenario_name

    metrics = {
        "scenario": scenario_name,
        "auc": roc_auc_score(y_test, y_prob),
        "threshold": fixed_threshold,
        "precision_eval": precision_score(y_test, final_preds, zero_division=0),
        "recall_eval": recall_score(y_test, final_preds, zero_division=0),
        "f1_eval": f1_score(y_test, final_preds, zero_division=0),
        "eval_in_test": int(sum(y_test)),
        "deploy_in_test": int(len(y_test) - sum(y_test)),
        "steering_vector": model.coef_[0]
    }

    return metrics, test_df

def run_evaluation_pipeline(df, scenario_list=None):
 
    
    if scenario_list is None:
        scenario_list = df["scenario"].unique()
        
    results_list = []
    all_test_dfs = []


    for scenario_name in scenario_list:
     
        metrics, test_subset_df = run_scenario_probe_final(df, scenario_name)
        
        if metrics:
            results_list.append(metrics)
        if test_subset_df is not None:
            all_test_dfs.append(test_subset_df)

   
    summary_df = pd.DataFrame(results_list)
    predictions_df = pd.concat(all_test_dfs, ignore_index=True)
    
    return summary_df, predictions_df


Para evaluar la capacidad de generalización del modelo, se implementa una validación cruzada tipo Leave-One-Scenario-Out (LOSO).

In [ ]:


def probe_held_out_scenario(df):
    raw_results_list = []
    raw_steering_vectors = {}
    
    scenarios = df["scenario"].unique()
    
    for held_out_scenario in scenarios:
      
        train_subset = df[df["scenario"] != held_out_scenario].copy()
        test_subset = df[df["scenario"] == held_out_scenario].copy()

        y_train_raw = create_eval_deploy_label(train_subset)
        y_test_raw = create_eval_deploy_label(test_subset)
        
        encoder = LabelEncoder()
        y_train = encoder.fit_transform(y_train_raw)
        
        try:
            y_test = encoder.transform(y_test_raw)
        except ValueError:
            raw_results_list.append({
                "held_out_scenario": held_out_scenario,
                "auc": np.nan,
                "status": "Unseen Test Label"
            })
            continue

     
        X_train = get_embedding_matrix(train_subset)
        X_test = get_embedding_matrix(test_subset)

        
        model = LogisticRegression(max_iter=5000, random_state=42)
        model.fit(X_train, y_train)

      
        raw_steering_vectors[held_out_scenario] = model.coef_[0]

    
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)

        raw_results_list.append({
            "held_out_scenario": held_out_scenario,
            "auc": auc,
            "train_size": len(train_subset),
            "test_size": len(test_subset),
            "status": "Success"
        })

    return pd.DataFrame(raw_results_list), raw_steering_vectors

loso_summary_df, loso_steering_vectors = probe_held_out_scenario(df)


print("\nResultados Left-One-Scenario-Out:")
print(loso_summary_df.sort_values(by="auc", ascending=False))

Como control se usa TF-IDF (Term Frequency-Inverse Document Frequency), debido a que permite aislar la información basada puramente en la presencia y frecuencia de términos dentro del prompt. Al comparar el desempeño de este baseline frente a nuestros probes lineales sobre embeddings, podemos determinar si la capacidad del modelo para distinguir el entorno de evaluación depende de rasgos superficiales del lenguaje o de una comprensión semántica del contexto de ejecución.

In [ ]:
      

def run_tfidf_probe(df, scenario_name):
    subset = df[df["scenario"] == scenario_name].copy()

    if subset.empty:
        return None

    y_raw = create_eval_deploy_label(subset)

    y = np.where(y_raw == "eval", 1, 0)


    texts = subset["response"].astype(str).values

    X_train_text, X_test_text, y_train, y_test = train_test_split(
        texts,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words=None,
        max_features=10000,
        ngram_range=(1, 2)
    )

    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    model = LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

  
    auc = roc_auc_score(y_test, y_prob)


    feature_names = np.array(
        vectorizer.get_feature_names_out()
    )

    coefficients = model.coef_[0]

    top_eval_idx = np.argsort(coefficients)[-20:][::-1]

    top_eval_features = pd.DataFrame({
        "feature": feature_names[top_eval_idx],
        "weight": coefficients[top_eval_idx]
    })

 
    top_deploy_idx = np.argsort(coefficients)[:20]

    top_deploy_features = pd.DataFrame({
        "feature": feature_names[top_deploy_idx],
        "weight": coefficients[top_deploy_idx]
    })

    return {
        "scenario": scenario_name,
        "tfidf_auc": round(auc, 4),
        "gt": y_test,        
        "prob_eval": y_prob,  
        "top_eval_features": top_eval_features,
        "top_deploy_features": top_deploy_features
    }



def run_tfidf_pipeline(df, scenario_list=None):
  
    if scenario_list is None:
        scenario_list = df["scenario"].unique()
        
    results_list = []
    

    
    for scenario_name in scenario_list:
       
        result = run_tfidf_probe(df, scenario_name)
        
        if result:
            results_list.append(result)
            

    summary_df = pd.DataFrame([
        {"scenario": r["scenario"], "tfidf_auc": r["tfidf_auc"]} 
        for r in results_list
    ])
    
    return summary_df, results_list
def print_tfidf_report(audit_details_list):
   
    for r in audit_details_list:
        print("\n" + "=" * 80)
        print(f"SCENARIO: {r['scenario']}")
        print("=" * 80)
        print("TOP EVAL FEATURES:", r["top_eval_features"])
        print("TOP DEPLOY FEATURES:", r["top_deploy_features"])




In [ ]:
summary_tfidf, details_tfidf = run_tfidf_pipeline(df)
summary_anonymized_tfidf, details_anonymized_tfidf = run_tfidf_pipeline(anonymized_df)

summary_orig, preds_orig = run_evaluation_pipeline(df)
summary_anonymized, preds_anon = run_evaluation_pipeline(anonymized_df)


comp_original = pd.merge(summary_orig[['scenario', 'auc']], 
                         summary_tfidf[['scenario', 'tfidf_auc']], 
                         on='scenario').rename(columns={'auc': 'probe_auc'})

comp_anonimizado = pd.merge(summary_anonymized[['scenario', 'auc']], 
                            summary_anonymized_tfidf[['scenario', 'tfidf_auc']], 
                            on='scenario').rename(columns={'auc': 'probe_auc'})


print("\n" + "="*80)
print("TABLA COMPARATIVA: Linear Probes vs TF-IDF | ORIGINAL")

print(comp_original.sort_values('probe_auc', ascending=False))
plot_probe_vs_tfidf(comp_original, 'Comparativa Dataset ORIGINAL')
print("\n" + "="*80)
print("TABLA COMPARATIVA: Linear Probes vs TF-IDF | MASKED ENTITIES")
print("="*80)
print(comp_anonimizado.sort_values('probe_auc', ascending=False))



plot_probe_vs_tfidf(comp_anonimizado, 'Comparativa Dataset MASKED ENTITIES')

